# Exparimentation with season functionality

In [1]:
import os

PROJECT_ROOT = "/Users/zanderholleran/Desktop/python_projects/mesa_lcc_model"
os.chdir(PROJECT_ROOT)

%pwd



# Enable auto-reloading of custom modules
%load_ext autoreload
%autoreload 2

In [2]:
from season.persons import SeasonPerson
from season.configs import ScheduleSpecs, SeasonConfig, DayParams, make_season_config, PopulationParams
from season.season_orchestrator import SeasonOrchestrator

import numpy as np
import pandas as pd
from scipy.stats import norm, lognorm, skewnorm, truncnorm, uniform


# Creating Schedules for TM 

In [ ]:


# Example ScheduleSpecs
# traffic_percentile_schedule = ScheduleSpecs(
#     mode='dist',
#     value=None,
#     dist=uniform(loc=60, scale=40),  # uniform distribution between 60 and 80
#     round_to_int=True
# )

traffic_percentile_schedule = ScheduleSpecs(
    mode='static',
    value=75
)

bus_interval_schedule = ScheduleSpecs(
    mode='static',
    value=30,  # static bus interval of 15 minutes
    dist=None
)

crashes_schedule = ScheduleSpecs(
    mode='static',
    value=0,
    #dist=norm(loc=5, scale=1)  # normal distribution for crashes per 100k VMT
)



In [3]:
traffic_percentile_schedule = ScheduleSpecs(
     mode='static',
    value=50,  
    dist=None
)

bus_interval_schedule = ScheduleSpecs(
    mode='static',
    value=30,  # static bus interval of 15 minutes
    dist=None
)

crashes_schedule = ScheduleSpecs(
    mode='static',
    value=4,  # static bus interval of 15 minutes
    dist=None # normal distribution for crashes per 100k VMT
)


print(traffic_percentile_schedule.realize( n_days=15))
print(crashes_schedule.realize( n_days=15))

[50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50]
[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]


# Defining the Population Perams

In [8]:
pop_params = PopulationParams(
    population_size=200,
    value_of_time=lognorm(s=0.64, scale=40/60),
    experience_weight_car=1.0,
    experience_weight_bus=skewnorm(6, loc=1.15, scale=0.3),
    prior_car=22.0,
    prior_bus=30.0,
    time_decay_rate=0.1,
    prior_weight=1.0,
    uncertainty_multiplier=1.0,
    travel_propensity=1.0,
)


config = make_season_config(
    season_id='two_week_example_001',
    run_description='Example config with custom schedules',
    seed=123,
    n_days=14,
    max_steps=10000,
    max_persons=200,
    collect_every_n=1000,
    start_hr=8,
    bus_capacity=30,
    road_path='data/roads/hw210_sl_and_curvs.parquet',
    ecs_path='data/vehicle_counts/expected_counts_seconds.csv',
    toll_mechanism='static',
    toll_params={'car': 20.0, 'bus': 0.0},
    canyon_closures_schedule=None,
    traffic_percentile_schedule=traffic_percentile_schedule,
    bus_interval_schedule=bus_interval_schedule,
    crashes_schedule=crashes_schedule, 
    population_params=pop_params
   
)

# light_config = make_season_config(
#     season_id='light_season_001',
#     run_description='Light config for testing',
#     seed=123,
#     n_days=15,
#     max_steps=3000,
#     max_persons=500,
#     collect_every_n=10,
#     road_path='data/roads/hw210_sl_and_curvs.parquet',
#     ecs_path='data/vehicle_counts/expected_counts_seconds.csv',
#     toll_mechanism='static',
#     toll_params={'car': 8.0, 'bus': 0.0},
#     population_params=pop_params,
#     crashes_schedule=crashes_schedule
    
# )

In [9]:
light_config

SeasonConfig(season_id='light_season_001', run_description='Light config for testing', seed=123, n_days=15, batch_run=True, max_steps=3000, max_persons=500, collect_every_n=10, start_hr=5, bus_capacity=30, road_path='data/roads/hw210_sl_and_curvs.parquet', ecs_path='data/vehicle_counts/expected_counts_seconds.csv', toll_mechanism='static', toll_params={'car': 8.0, 'bus': 0.0}, day_params=[DayParams(day_index=0, day_seed=123, traffic_percentile=50, bus_interval=30, crashes_per_100k_vmt_input=4.0, canyon_closures=None), DayParams(day_index=1, day_seed=124, traffic_percentile=50, bus_interval=30, crashes_per_100k_vmt_input=4.0, canyon_closures=None), DayParams(day_index=2, day_seed=125, traffic_percentile=50, bus_interval=30, crashes_per_100k_vmt_input=4.0, canyon_closures=None), DayParams(day_index=3, day_seed=126, traffic_percentile=50, bus_interval=30, crashes_per_100k_vmt_input=4.0, canyon_closures=None), DayParams(day_index=4, day_seed=127, traffic_percentile=50, bus_interval=30, cra

In [11]:
# Example usage of SeasonOrchestrator with example_config
orchestrator = SeasonOrchestrator(season_config=config, output_dir="data/season_outputs",)
orchestrator.run_season()



Toll mechanism: static with params {'car': 20.0, 'bus': 0.0}


Simulating:   7%|▋         | 728/10000 [00:00<00:05, 1565.13step/s]

crash at step 608


Simulating: 100%|██████████| 10000/10000 [00:29<00:00, 341.71step/s]


Reached max step count (10000). Stopping model.
Number of bus riders 146 vs car riders 54
Toll mechanism: static with params {'car': 20.0, 'bus': 0.0}


...

200 people generated stopping model.
Number of bus riders 68 vs car riders 132
Toll mechanism: static with params {'car': 20.0, 'bus': 0.0}


Simulating:   6%|▌         | 601/10000 [00:00<00:09, 1005.40step/s]

crash at step 468


Simulating:  60%|██████    | 6046/10000 [00:13<00:08, 440.22step/s]

200 people generated stopping model.
Number of bus riders 70 vs car riders 130


In [ ]:
pd.read_parquet("/Users/zanderholleran/Desktop/python_projects/mesa_lcc_model/data/season_outputs/example_season_001/day_1_model_ts.parquet")